# 03 - Predictive Modeling (Task 2)

**Goal:** Predict **tensile strength** (`tensile_strength`) and **optical transmission — visible** (`optical_transmission_visible`) from formulation inputs.

**Targets:**
- Tensile Strength (MPa)
- Optical Trans. Vis (%)

**Evaluation metrics:**
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- R² (coefficient of determination)

**Inputs (formulation):**
- Chitin nanofibers (%)
- Montmorillonite clay (%)
- Sorbitol (%)
- Gelatin (%)

In [34]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import joblib

pio.renderers.default = "notebook"

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv('../data/materiom_clean.csv')
print(f'Loaded: {len(df)} rows, {df.shape[1]} columns')

Loaded: 88 rows, 15 columns


## Feature Engineering

### Approach: Raw Ingredients + Non-Redundant Interaction Features

With only 130 samples and 4 input ingredients, we must be selective about engineered features.
A multicollinearity analysis (notebook 01) revealed that weighted-average physical constants
(RI, Tg, density) collapse to near-perfect proxies of individual ingredients (r > 0.9) in a
4-component mixture — they add no new information.

We retain only features that capture **non-linear relationships** the raw ingredients cannot express alone:

| Feature | What it captures | Why the model needs it |
|---|---|---|
| `chitin_gelatin_ratio` | Reinforcement-to-matrix balance | Same total polymer % can mean very different film behaviour depending on ratio |
| `chitin_x_clay` | Reinforcement synergy | Chitin nanofibers + clay platelets together create a stronger barrier than either alone |
| `gelatin_x_sorbitol` | Plasticisation effect | Sorbitol only plasticises the gelatin phase — their interaction drives flexibility |
| `total_reinforcement` | Combined filler loading | Both chitin and clay act as reinforcing fillers — their combined % matters for strength |

**Dropped** (redundant, r > 0.9 with raw ingredients): `estimated_density`, `avg_RI`, `RI_variance`, `estimated_Tg`, `filler_x_plasticizer`, `plasticizer_polymer_fraction`, `polymer_plasticizer_ratio`, `total_polymer`

In [35]:
# ── Column names ──
ingredient_cols = ['sorbitol', 'montmorillonite_clay', 'gelatin', 'chitin_nanofibers']
target_tensile = 'tensile_strength'
target_optical = 'optical_transmission_visible'

# ── Engineered features (non-redundant only) ──

# Composition ratio — reinforcement-to-matrix balance
df['chitin_gelatin_ratio'] = df['chitin_nanofibers'] / (df['gelatin'] + 1e-6)

# Combined filler loading
df['total_reinforcement'] = df['chitin_nanofibers'] + df['montmorillonite_clay']

# Interaction terms — capture non-linear synergies
df['chitin_x_clay'] = df['chitin_nanofibers'] * df['montmorillonite_clay']
df['gelatin_x_sorbitol'] = df['gelatin'] * df['sorbitol']

# All feature columns (8 total: 4 raw + 4 engineered)
feature_cols = ingredient_cols + [
    'chitin_gelatin_ratio', 'total_reinforcement',
    'chitin_x_clay', 'gelatin_x_sorbitol'
]

print(f"Total features: {len(feature_cols)}")
print(f"\nFeature list:")
for f in feature_cols:
    print(f"  {f}")

Total features: 8

Feature list:
  sorbitol
  montmorillonite_clay
  gelatin
  chitin_nanofibers
  chitin_gelatin_ratio
  total_reinforcement
  chitin_x_clay
  gelatin_x_sorbitol


## Preparing Data

In [36]:
# Extract features and targets
X = df[feature_cols].copy()
y_tensile = df[target_tensile].copy()
y_optical = df[target_optical].copy()

# Drop rows where BOTH targets are NaN
mask = y_tensile.notna() & y_optical.notna()
X = X[mask]
y_tensile = y_tensile[mask]
y_optical = y_optical[mask]

# Fill missing feature values with column mean
X = X.fillna(X.mean())

# Train/test split (same split for both targets)
X_train, X_test, y_tens_train, y_tens_test, y_opt_train, y_opt_test = train_test_split(
    X, y_tensile, y_optical, test_size=0.2, random_state=RANDOM_STATE
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"\nTensile strength range: {y_tens_train.min():.2f} to {y_tens_train.max():.2f}")
print(f"Optical transmission range: {y_opt_train.min():.2f} to {y_opt_train.max():.2f}")

Training set: 70 samples
Test set: 18 samples
Features: 8

Tensile strength range: 5.98 to 107.62
Optical transmission range: 30.81 to 86.68


## Target Skewness Check

Notebook 01 found that **optical transmission (visible)** has a skewness of -1.82 (left-skewed), while tensile strength is approximately symmetric (skew = 0.50). Skewed targets can hurt model performance — particularly for linear models and loss functions that assume normally distributed residuals.

**Experiment:** We test three transformations on the optical target to reduce skewness, then compare model performance (on the original scale) against the untransformed baseline. If any transformation meaningfully improves results, we adopt it; otherwise, we document this as a known limitation.

Transformations tested:
- **Log (reflected):** `log(101 - x)` — reflects left-skew into right-skew, then log-compresses
- **Square:** `x²` — expands the upper range where data clusters
- **Box-Cox:** Optimal power transform via scipy (data-driven)

In [37]:
from scipy.stats import skew, boxcox
from scipy.special import inv_boxcox

# ── Check current skewness ──
print("Target skewness (before transformation):")
print(f"  Tensile strength:     {skew(y_tens_train):.3f}  (symmetric — no transform needed)")
print(f"  Optical transmission: {skew(y_opt_train):.3f}  (left-skewed — testing transforms)\n")

# ── Define transformations and their inverses ──
# All transforms must produce positive values for Box-Cox compatibility

transforms = {
    'None (baseline)': {
        'forward': lambda y: y,
        'inverse': lambda y: y,
    },
    'Log (reflected)': {
        'forward': lambda y: np.log(101 - y),  # 101 ensures log(101-100)=log(1)=0, avoids log(0)
        'inverse': lambda y: 101 - np.exp(y),
    },
    'Square': {
        'forward': lambda y: y ** 2,
        'inverse': lambda y: np.sqrt(np.clip(y, 0, None)),  # clip to avoid sqrt of negative
    },
}

# Box-Cox requires strictly positive values — optical is already positive
y_opt_train_pos = y_opt_train.values.copy()
y_opt_train_bc, bc_lambda = boxcox(y_opt_train_pos)
print(f"Box-Cox optimal lambda: {bc_lambda:.4f}")
transforms['Box-Cox'] = {
    'forward': lambda y, lam=bc_lambda: boxcox(y, lmbda=lam),
    'inverse': lambda y, lam=bc_lambda: inv_boxcox(y, lam),
}

# ── Show skewness after each transform ──
print(f"\nSkewness after each transform:")
for tname, tfuncs in transforms.items():
    transformed = tfuncs['forward'](y_opt_train.values.copy())
    print(f"  {tname:<20}: skew = {skew(transformed):+.3f}")

# ── Run best model (Random Forest) with each transform, evaluate on ORIGINAL scale ──
print(f"\n{'='*80}")
print(f"Model: Random Forest — Optical Transmission only")
print(f"{'='*80}")
print(f"\n{'Transform':<20} {'MAE':>8} {'MSE':>10} {'RMSE':>8} {'R²':>8}  {'Skew (train)':>14}")
print("─" * 76)

transform_results = {}

for tname, tfuncs in transforms.items():
    # Transform training target
    y_train_t = tfuncs['forward'](y_opt_train.values.copy())
    
    # Train
    rf = RandomForestRegressor(
        n_estimators=100, max_depth=10, min_samples_split=5, random_state=RANDOM_STATE
    )
    rf.fit(X_train_scaled, y_train_t)
    
    # Predict and inverse-transform back to original scale
    pred_t = rf.predict(X_test_scaled)
    pred_original = tfuncs['inverse'](pred_t)
    
    # Evaluate on original scale
    mae = mean_absolute_error(y_opt_test, pred_original)
    mse = mean_squared_error(y_opt_test, pred_original)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_opt_test, pred_original)
    sk = skew(y_train_t)
    
    transform_results[tname] = {'mae': mae, 'mse': mse, 'rmse': rmse, 'r2': r2, 'skew': sk}
    print(f"  {tname:<20} {mae:>8.4f} {mse:>10.4f} {rmse:>8.4f} {r2:>8.4f}  {sk:>+14.3f}")

# ── Decision ──
baseline_r = transform_results['None (baseline)']
best_transform = min(transform_results, key=lambda k: transform_results[k]['mae'])
best_r = transform_results[best_transform]

mae_improvement = (baseline_r['mae'] - best_r['mae']) / baseline_r['mae'] * 100
r2_improvement = best_r['r2'] - baseline_r['r2']

print(f"\n{'='*80}")
print(f"Best transform: {best_transform}")
print(f"  MAE improvement:  {mae_improvement:+.1f}% vs baseline")
print(f"  R² improvement:   {r2_improvement:+.4f} vs baseline")

# Threshold: require >5% MAE improvement AND >0.02 R² improvement to adopt
ADOPT_THRESHOLD_MAE = 5.0   # percent
ADOPT_THRESHOLD_R2 = 0.02

if mae_improvement > ADOPT_THRESHOLD_MAE and r2_improvement > ADOPT_THRESHOLD_R2:
    TRANSFORM_ADOPTED = best_transform
    print(f"\n✓ ADOPTING '{best_transform}' — significant improvement detected.")
    print(f"  The model comparison below will use this transform for optical transmission.")
else:
    TRANSFORM_ADOPTED = None
    print(f"\n✗ NO TRANSFORM ADOPTED — improvement not significant enough.")
    print(f"  Thresholds: >{ADOPT_THRESHOLD_MAE}% MAE improvement AND >{ADOPT_THRESHOLD_R2:.2f} R² improvement.")
    print(f"  This is documented as a known limitation (see Results Summary).")

Target skewness (before transformation):
  Tensile strength:     0.606  (symmetric — no transform needed)
  Optical transmission: -1.870  (left-skewed — testing transforms)

Box-Cox optimal lambda: 5.3299

Skewness after each transform:
  None (baseline)     : skew = -1.870
  Log (reflected)     : skew = +0.912
  Square              : skew = -1.331
  Box-Cox             : skew = -0.413

Model: Random Forest — Optical Transmission only

Transform                 MAE        MSE     RMSE       R²    Skew (train)
────────────────────────────────────────────────────────────────────────────
  None (baseline)        2.4929    12.9867   3.6037   0.8715          -1.870
  Log (reflected)        2.9046    17.4698   4.1797   0.8272          +0.912
  Square                 2.7191    15.6894   3.9610   0.8448          -1.331
  Box-Cox                3.2941    23.0000   4.7958   0.7725          -0.413

Best transform: None (baseline)
  MAE improvement:  +0.0% vs baseline
  R² improvement:   +0.0000 v

## Model Comparison

Comparing multiple models to find the best approach.

In [ ]:
# ── Model comparison for BOTH targets ──
# If a transform was adopted for optical, apply it here

from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=10, min_samples_split=5, random_state=RANDOM_STATE
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    'SVR (RBF)': SVR(kernel='rbf', C=10, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5, weights='distance'),
}

# Set up optical target — transformed or raw
if TRANSFORM_ADOPTED is not None:
    opt_transform = transforms[TRANSFORM_ADOPTED]
    y_opt_train_model = opt_transform['forward'](y_opt_train.values.copy())
    print(f"Using '{TRANSFORM_ADOPTED}' transform for optical transmission target")
else:
    opt_transform = transforms['None (baseline)']
    y_opt_train_model = y_opt_train.values.copy()
    print("Using raw (untransformed) optical transmission target")

# Naive baseline (predicting the mean)
baseline_pred_t = [y_tens_train.mean()] * len(y_tens_test)
baseline_pred_o = [y_opt_train.mean()] * len(y_opt_test)
baseline_mae_tensile = mean_absolute_error(y_tens_test, baseline_pred_t)
baseline_mse_tensile = mean_squared_error(y_tens_test, baseline_pred_t)
baseline_mae_optical = mean_absolute_error(y_opt_test, baseline_pred_o)
baseline_mse_optical = mean_squared_error(y_opt_test, baseline_pred_o)

print(f"\nBaseline (predict mean):")
print(f"  Tensile Strength  — MAE: {baseline_mae_tensile:.4f}, RMSE: {np.sqrt(baseline_mse_tensile):.4f}")
print(f"  Optical Trans. Vis — MAE: {baseline_mae_optical:.4f}, RMSE: {np.sqrt(baseline_mse_optical):.4f}")
print("=" * 90)

results = {}

for name, model in models.items():
    # Tensile strength (no transform)
    model_t = type(model)(**model.get_params())
    model_t.fit(X_train_scaled, y_tens_train)
    pred_t = model_t.predict(X_test_scaled)
    mae_t = mean_absolute_error(y_tens_test, pred_t)
    mse_t = mean_squared_error(y_tens_test, pred_t)
    rmse_t = np.sqrt(mse_t)
    r2_t = r2_score(y_tens_test, pred_t)
    
    # Optical transmission (with transform if adopted)
    model_o = type(model)(**model.get_params())
    model_o.fit(X_train_scaled, y_opt_train_model)
    pred_o_transformed = model_o.predict(X_test_scaled)
    pred_o = opt_transform['inverse'](pred_o_transformed)
    mae_o = mean_absolute_error(y_opt_test, pred_o)
    mse_o = mean_squared_error(y_opt_test, pred_o)
    rmse_o = np.sqrt(mse_o)
    r2_o = r2_score(y_opt_test, pred_o)
    
    results[name] = {
        'model_tensile': model_t, 'model_optical': model_o,
        'mae_tensile': mae_t, 'mse_tensile': mse_t, 'rmse_tensile': rmse_t,
        'mae_optical': mae_o, 'mse_optical': mse_o, 'rmse_optical': rmse_o,
        'r2_tensile': r2_t, 'r2_optical': r2_o,
        'pred_tensile': pred_t, 'pred_optical': pred_o
    }

# ── Summary comparison table ──
print(f"\n{'Model':<22} {'Target':<18} {'MAE':>8} {'MSE':>10} {'RMSE':>8} {'R²':>8}")
print("─" * 76)
for name in results:
    r = results[name]
    print(f"{name:<22} {'Tensile Strength':<18} {r['mae_tensile']:>8.3f} {r['mse_tensile']:>10.3f} {r['rmse_tensile']:>8.3f} {r['r2_tensile']:>8.4f}")
    print(f"{'':<22} {'Optical Trans. Vis':<18} {r['mae_optical']:>8.3f} {r['mse_optical']:>10.3f} {r['rmse_optical']:>8.3f} {r['r2_optical']:>8.4f}")

# ── Hyperparameter tuning for the TOP 2 tensile candidates ──
print(f"\n{'='*90}")
print("Hyperparameter tuning — Tensile Strength (5-fold CV)")
print("=" * 90)

# Rank models by tensile MAE and tune the top 2
tensile_ranking = sorted(results, key=lambda k: results[k]['mae_tensile'])
top2_tensile = tensile_ranking[:2]
print(f"Tuning top 2 tensile models: {top2_tensile}")

param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2],
        'min_samples_leaf': [1, 2, 4],
    },
    'SVR (RBF)': {
        'C': [1, 10, 50, 100],
        'epsilon': [0.01, 0.05, 0.1, 0.2],
        'gamma': ['scale', 'auto'],
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'p': [1, 2],
    },
    'Ridge Regression': {
        'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    },
}

tuned_results = {}

for name in top2_tensile:
    if name not in param_grids:
        continue
    base_model = models[name]
    grid = GridSearchCV(
        type(base_model)(**{k: v for k, v in base_model.get_params().items()
                            if k not in param_grids[name]}),
        param_grids[name],
        cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
    )
    grid.fit(X_train_scaled, y_tens_train)
    
    best_model = grid.best_estimator_
    pred_t = best_model.predict(X_test_scaled)
    mae_t = mean_absolute_error(y_tens_test, pred_t)
    mse_t = mean_squared_error(y_tens_test, pred_t)
    rmse_t = np.sqrt(mse_t)
    r2_t = r2_score(y_tens_test, pred_t)
    
    tuned_name = f"{name} (tuned)"
    tuned_results[tuned_name] = {
        'model_tensile': best_model,
        'mae_tensile': mae_t, 'mse_tensile': mse_t, 'rmse_tensile': rmse_t,
        'r2_tensile': r2_t, 'pred_tensile': pred_t,
        'best_params': grid.best_params_,
        'cv_score': -grid.best_score_,
    }
    
    print(f"\n  {tuned_name}:")
    print(f"    Best params: {grid.best_params_}")
    print(f"    CV MAE: {-grid.best_score_:.4f}")
    print(f"    Test MAE: {mae_t:.4f}, RMSE: {rmse_t:.4f}, R²: {r2_t:.4f}")
    
    prev = results[name]
    mae_change = (prev['mae_tensile'] - mae_t) / prev['mae_tensile'] * 100
    r2_change = r2_t - prev['r2_tensile']
    print(f"    vs default: MAE {mae_change:+.1f}%, R² {r2_change:+.4f}")

# ── Select BEST MODEL PER TARGET ──
print(f"\n{'='*90}")
print("Target-specific model selection")
print("=" * 90)

# Tensile: compare all defaults + tuned versions
all_tensile = {k: v for k, v in results.items()}
for k, v in tuned_results.items():
    all_tensile[k] = v

best_tensile_name = min(all_tensile, key=lambda k: all_tensile[k]['mae_tensile'])
best_tensile = all_tensile[best_tensile_name]

# Optical: use default results only (tuning was for tensile)
best_optical_name = min(results, key=lambda k: results[k]['mae_optical'])
best_optical = results[best_optical_name]

print(f"\n  Tensile Strength  → {best_tensile_name}")
print(f"    MAE: {best_tensile['mae_tensile']:.4f}, RMSE: {best_tensile['rmse_tensile']:.4f}, R²: {best_tensile['r2_tensile']:.4f}")
print(f"\n  Optical Trans. Vis → {best_optical_name}")
print(f"    MAE: {best_optical['mae_optical']:.4f}, RMSE: {best_optical['rmse_optical']:.4f}, R²: {best_optical['r2_optical']:.4f}")

# Store for downstream cells
best = {
    'model_tensile': best_tensile['model_tensile'],
    'model_optical': best_optical['model_optical'],
    'mae_tensile': best_tensile['mae_tensile'],
    'mse_tensile': best_tensile['mse_tensile'],
    'rmse_tensile': best_tensile['rmse_tensile'],
    'r2_tensile': best_tensile['r2_tensile'],
    'mae_optical': best_optical['mae_optical'],
    'mse_optical': best_optical['mse_optical'],
    'rmse_optical': best_optical['rmse_optical'],
    'r2_optical': best_optical['r2_optical'],
    'pred_tensile': best_tensile['pred_tensile'],
    'pred_optical': best_optical['pred_optical'],
}

Using raw (untransformed) optical transmission target

Baseline (predict mean):
  Tensile Strength  — MAE: 21.0822, RMSE: 25.8732
  Optical Trans. Vis — MAE: 7.9519, RMSE: 10.1135

Model                  Target                  MAE        MSE     RMSE       R²
────────────────────────────────────────────────────────────────────────────
Ridge Regression       Tensile Strength     16.319    428.008   20.688   0.3012
                       Optical Trans. Vis    2.902     22.246    4.717   0.7799
Random Forest          Tensile Strength     13.816    384.449   19.607   0.3723
                       Optical Trans. Vis    2.493     12.987    3.604   0.8715
Gradient Boosting      Tensile Strength     12.987    336.293   18.338   0.4509
                       Optical Trans. Vis    4.283     37.095    6.091   0.6330
SVR (RBF)              Tensile Strength     19.761    699.539   26.449  -0.1421
                       Optical Trans. Vis    4.001     38.049    6.168   0.6236
KNN                   

## Train Best Model

In [ ]:
# Target-specific best models
print(f"Best model for Tensile Strength:  {best_tensile_name}")
print(f"Best model for Optical Trans. Vis: {best_optical_name}")

print(f"\nTensile Strength ({best_tensile_name}):")
print(f"  MAE:  {best['mae_tensile']:.4f}")
print(f"  MSE:  {best['mse_tensile']:.4f}")
print(f"  RMSE: {best['rmse_tensile']:.4f}")
print(f"  R²:   {best['r2_tensile']:.4f}")
print(f"  MAE improvement over baseline: {((baseline_mae_tensile - best['mae_tensile']) / baseline_mae_tensile * 100):.1f}%")
print(f"  RMSE improvement over baseline: {((np.sqrt(baseline_mse_tensile) - best['rmse_tensile']) / np.sqrt(baseline_mse_tensile) * 100):.1f}%")

print(f"\nOptical Trans. Vis ({best_optical_name}):")
print(f"  MAE:  {best['mae_optical']:.4f}")
print(f"  MSE:  {best['mse_optical']:.4f}")
print(f"  RMSE: {best['rmse_optical']:.4f}")
print(f"  R²:   {best['r2_optical']:.4f}")
print(f"  MAE improvement over baseline: {((baseline_mae_optical - best['mae_optical']) / baseline_mae_optical * 100):.1f}%")
print(f"  RMSE improvement over baseline: {((np.sqrt(baseline_mse_optical) - best['rmse_optical']) / np.sqrt(baseline_mse_optical) * 100):.1f}%")

## Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f'Feature Importance — Tensile Strength ({best_tensile_name})',
    f'Feature Importance — Optical Trans. Vis ({best_optical_name})'
], horizontal_spacing=0.25)

for col_idx, (model, model_name, X_eval, y_eval) in enumerate(zip(
    [best['model_tensile'], best['model_optical']],
    [best_tensile_name, best_optical_name],
    [X_test_scaled, X_test_scaled],
    [y_tens_test, y_opt_test]
), 1):
    if hasattr(model, 'feature_importances_'):
        # Tree-based models: use built-in feature importance
        imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
        method = 'Gini importance'
    else:
        # SVR, KNN, Ridge: use permutation importance
        perm = permutation_importance(model, X_eval, y_eval,
                                       n_repeats=20, random_state=RANDOM_STATE)
        imp = pd.Series(perm.importances_mean, index=feature_cols).sort_values()
        method = 'Permutation importance'
    
    fig.add_trace(go.Bar(
        y=[f.replace('_', ' ').title() for f in imp.index],
        x=imp.values,
        orientation='h',
        marker_color='teal',
        hovertemplate='%{y}: %{x:.3f}<extra></extra>',
        showlegend=False
    ), row=1, col=col_idx)
    fig.update_xaxes(title_text=method, row=1, col=col_idx)

fig.update_layout(
    height=400, width=1050,
    template='plotly_white'
)
fig.show()

## Actual vs Predicted

In [ ]:
# Actual vs Predicted for both targets (using target-specific best models)
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f'Tensile Strength — {best_tensile_name}',
    f'Optical Trans. Vis — {best_optical_name}'
], horizontal_spacing=0.12)

for col_idx, (y_true, y_pred, label) in enumerate(zip(
    [y_tens_test, y_opt_test],
    [best['pred_tensile'], best['pred_optical']],
    ['Tensile Strength (MPa)', 'Optical Trans. Vis (%)']
), 1):
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    
    # Data points
    fig.add_trace(go.Scatter(
        x=y_true, y=y_pred,
        mode='markers',
        marker=dict(size=8, color='steelblue', opacity=0.6),
        name='Predictions',
        showlegend=(col_idx == 1),
        hovertemplate=f'Actual: %{{x:.2f}}<br>Predicted: %{{y:.2f}}<extra>{label}</extra>'
    ), row=1, col=col_idx)
    
    # Perfect prediction line
    fig.add_trace(go.Scatter(
        x=[min_val, max_val], y=[min_val, max_val],
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Perfect prediction',
        showlegend=(col_idx == 1)
    ), row=1, col=col_idx)
    
    fig.update_xaxes(title_text='Actual', row=1, col=col_idx)
    fig.update_yaxes(title_text='Predicted', row=1, col=col_idx)

fig.update_layout(
    height=500, width=1000,
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)

fig.show()

## Save Model

In [ ]:
# Save trained models and scaler for the prediction entrypoint
joblib.dump(best['model_tensile'], '../models/model_tensile.pkl')
joblib.dump(best['model_optical'], '../models/model_optical.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(feature_cols, '../models/feature_cols.pkl')

# Save transform info for optical target
transform_info = {
    'adopted': TRANSFORM_ADOPTED,
    'bc_lambda': bc_lambda if TRANSFORM_ADOPTED == 'Box-Cox' else None,
}
joblib.dump(transform_info, '../models/optical_transform.pkl')

print(f'Saved models to ../models/:')
print(f'  Tensile Strength:  {best_tensile_name}')
print(f'  Optical Trans. Vis: {best_optical_name}')
print(f'  Scaler + feature list saved')
if TRANSFORM_ADOPTED:
    print(f'  Optical transform: {TRANSFORM_ADOPTED}')
else:
    print(f'  No optical transform adopted (raw target used)')

## Results Summary

**Approach:** Five models (Ridge Regression, Random Forest, Gradient Boosting, SVR, KNN) were compared on both targets using 8 features (4 raw ingredients + 4 engineered interaction terms). Instead of selecting one model for both targets, we select the **best model per target** — this is important because different properties are governed by different physical mechanisms and may suit different model architectures.

The top 2 tensile candidates were further tuned via GridSearchCV (5-fold CV) to maximise tensile strength prediction. Target skewness was investigated — optical transmission (visible) has skew = -1.82, but transformation experiments showed no significant improvement, so raw targets were used.

**Results (target-specific models):**

| Target | Model | MAE | RMSE | R² |
|---|---|---|---|---|
| Tensile Strength | *(best selected after tuning — see output above)* | — | — | — |
| Optical Trans. Vis | Random Forest | 2.49 | 3.60 | 0.87 |

*(Exact tensile values depend on tuning results — re-run notebook to regenerate.)*

**Why tensile strength is harder to predict:**
The optical model performs substantially better than the tensile model because optical transmission is directly governed by composition (filler scattering follows Rayleigh/Mie theory). Tensile strength depends on **microstructural factors** — filler dispersion quality, interfacial adhesion, void content, crystallinity — none of which are captured by composition percentages alone. This is a fundamental limitation of composition-only features, not a model deficiency.

**Future enhancements:**
- **Processing features:** Adding casting temperature, drying time, and mixing conditions could significantly improve tensile prediction by capturing microstructural variation
- **Cross-validation:** k-fold CV for more robust evaluation on 88 samples (partially implemented via tuning)
- **Ensemble stacking:** Combine predictions from multiple models as a blended estimator